### importing libraries

In [2]:
from pyspark.sql import SparkSession
from pathlib import Path

### Defining the Dataset Path

In [3]:
DATA_PATH = Path("data/ml-latest-small")

### Defining spark session

In [4]:
spark = SparkSession.builder.appName("MovieLens Analysis").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/24 17:12:34 WARN Utils: Your hostname, raj-s, resolves to a loopback address: 127.0.1.1; using 10.137.86.44 instead (on interface wlp0s20f3)
26/03/24 17:12:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/24 17:12:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
candidate_paths = [
	DATA_PATH,
	Path.cwd() / DATA_PATH,
	Path.cwd().parent / DATA_PATH,
]

resolved_data_path = next((p for p in candidate_paths if (p / "movies.csv").exists()), None)
if resolved_data_path is None:
	raise FileNotFoundError(f"Could not find movies.csv in any of: {candidate_paths}")

movies = spark.read.csv(str(resolved_data_path / "movies.csv"), header=True, inferSchema=True)
ratings = spark.read.csv(str(resolved_data_path / "ratings.csv"), header=True, inferSchema=True)
tags = spark.read.csv(str(resolved_data_path / "tags.csv"), header=True, inferSchema=True)

### Exploring data

In [6]:
movies.show(5)
ratings.show(5)
tags.show(5)

+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
+-------+--------------------+--------------------+
only showing top 5 rows
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
|     1|     47|   5.0|964983815|
|     1|     50|   5.0|964982931|
+------+-------+------+---------+
only showing top 5 rows
+------+-------+---------------+----------+
|userId|movieId|            tag| timestamp|
+------+-------+---------------+----------+
|     2|  60756|          funny|1445714994|
| 

In [7]:
movies.printSchema()
ratings.printSchema()
tags.printSchema()

root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- tag: string (nullable = true)
 |-- timestamp: integer (nullable = true)



In [8]:
print("Movies count:", movies.count())
print("Ratings count:", ratings.count())
print("Tags count:", tags.count())

Movies count: 9742
Ratings count: 100836
Tags count: 3683


#### Joining the tables

In [9]:
df = ratings.join(movies, on="movieId", how="inner")
df.show(5)

+-------+------+------+---------+--------------------+--------------------+
|movieId|userId|rating|timestamp|               title|              genres|
+-------+------+------+---------+--------------------+--------------------+
|      1|     1|   4.0|964982703|    Toy Story (1995)|Adventure|Animati...|
|      3|     1|   4.0|964981247|Grumpier Old Men ...|      Comedy|Romance|
|      6|     1|   4.0|964982224|         Heat (1995)|Action|Crime|Thri...|
|     47|     1|   5.0|964983815|Seven (a.k.a. Se7...|    Mystery|Thriller|
|     50|     1|   5.0|964982931|Usual Suspects, T...|Crime|Mystery|Thr...|
+-------+------+------+---------+--------------------+--------------------+
only showing top 5 rows


#### Checking for null values

In [10]:
from pyspark.sql.functions import col, sum

df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

+-------+------+------+---------+-----+------+
|movieId|userId|rating|timestamp|title|genres|
+-------+------+------+---------+-----+------+
|      0|     0|     0|        0|    0|     0|
+-------+------+------+---------+-----+------+



> No null values so we can skip dropping nulls

In [11]:
df.columns

['movieId', 'userId', 'rating', 'timestamp', 'title', 'genres']

### Data cleaning

In [12]:
from pyspark.sql.functions import split, explode

df = df.withColumn("genres", explode(split(df["genres"], "\\|")))

In [13]:
df.select("title", "genres").show(10, False)

+-----------------------+---------+
|title                  |genres   |
+-----------------------+---------+
|Toy Story (1995)       |Adventure|
|Toy Story (1995)       |Animation|
|Toy Story (1995)       |Children |
|Toy Story (1995)       |Comedy   |
|Toy Story (1995)       |Fantasy  |
|Grumpier Old Men (1995)|Comedy   |
|Grumpier Old Men (1995)|Romance  |
|Heat (1995)            |Action   |
|Heat (1995)            |Crime    |
|Heat (1995)            |Thriller |
+-----------------------+---------+
only showing top 10 rows


In [14]:
df.printSchema()
df.show(5)

root
 |-- movieId: integer (nullable = true)
 |-- userId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = false)

+-------+------+------+---------+----------------+---------+
|movieId|userId|rating|timestamp|           title|   genres|
+-------+------+------+---------+----------------+---------+
|      1|     1|   4.0|964982703|Toy Story (1995)|Adventure|
|      1|     1|   4.0|964982703|Toy Story (1995)|Animation|
|      1|     1|   4.0|964982703|Toy Story (1995)| Children|
|      1|     1|   4.0|964982703|Toy Story (1995)|   Comedy|
|      1|     1|   4.0|964982703|Toy Story (1995)|  Fantasy|
+-------+------+------+---------+----------------+---------+
only showing top 5 rows


### Top rated movies

In [15]:
from pyspark.sql.functions import avg, count

top_movies = df.groupBy("title") \
    .agg(avg("rating").alias("avg_rating"), count("rating").alias("num_ratings")) \
    .filter("num_ratings > 50") \
    .orderBy("avg_rating", ascending=False)

top_movies.show(10, False)

+-------------------------------------------------------------+-----------------+-----------+
|title                                                        |avg_rating       |num_ratings|
+-------------------------------------------------------------+-----------------+-----------+
|Shawshank Redemption, The (1994)                             |4.429022082018927|634        |
|Sunset Blvd. (a.k.a. Sunset Boulevard) (1950)                |4.333333333333333|81         |
|Double Indemnity (1944)                                      |4.323529411764706|51         |
|Philadelphia Story, The (1940)                               |4.310344827586207|87         |
|Once Upon a Time in the West (C'era una volta il West) (1968)|4.305555555555555|54         |
|Lawrence of Arabia (1962)                                    |4.3              |135        |
|Godfather, The (1972)                                        |4.2890625        |384        |
|Harold and Maude (1971)                                    

### Genre based avrerage ratinmgs

In [16]:
genre_ratings = df.groupBy("genres") \
    .agg(avg("rating").alias("avg_rating")) \
    .orderBy("avg_rating", ascending=False)

genre_ratings.show()

+------------------+------------------+
|            genres|        avg_rating|
+------------------+------------------+
|         Film-Noir| 3.920114942528736|
|               War|   3.8082938876312|
|       Documentary| 3.797785069729286|
|             Crime| 3.658293867274144|
|             Drama|3.6561844113718758|
|           Mystery| 3.632460255407871|
|         Animation|3.6299370349170004|
|              IMAX| 3.618335343787696|
|           Western| 3.583937823834197|
|           Musical|3.5636781053649105|
|         Adventure|3.5086089151939075|
|           Romance|3.5065107040388437|
|          Thriller|3.4937055799183425|
|           Fantasy|3.4910005070136894|
|(no genres listed)|3.4893617021276597|
|            Sci-Fi| 3.455721162210752|
|            Action| 3.447984331646809|
|          Children| 3.412956125108601|
|            Comedy|3.3847207640898267|
|            Horror| 3.258195034974626|
+------------------+------------------+



### Most Active users

In [17]:
active_users = df.groupBy("userId") \
    .agg(count("rating").alias("num_ratings")) \
    .orderBy("num_ratings", ascending=False)

active_users.show(10)

+------+-----------+
|userId|num_ratings|
+------+-----------+
|   414|       6616|
|   599|       6266|
|   474|       4739|
|   448|       4702|
|   380|       3761|
|   274|       3716|
|   610|       3714|
|    68|       3447|
|   249|       3028|
|   288|       2779|
+------+-----------+
only showing top 10 rows


In [18]:
tag_df = tags.join(movies, on="movieId", how="inner")

In [19]:
from pyspark.sql.functions import count

tag_counts = tag_df.groupBy("tag") \
    .agg(count("*").alias("count")) \
    .orderBy("count", ascending=False)

tag_counts.show(10, False)

+-----------------+-----+
|tag              |count|
+-----------------+-----+
|In Netflix queue |131  |
|atmospheric      |36   |
|superhero        |24   |
|thought-provoking|24   |
|surreal          |23   |
|Disney           |23   |
|funny            |23   |
|religion         |22   |
|psychology       |21   |
|quirky           |21   |
+-----------------+-----+
only showing top 10 rows


In [20]:
top_movies.write.csv("output/top_movies", header=True)
genre_ratings.write.csv("output/genre_ratings", header=True)
active_users.write.csv("output/active_users", header=True)
tag_counts.write.csv("output/tag_counts", header=True)